In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog = "retail_catalog1"
schema = "bronze"

source_path = "/Volumes/retail_catalog1/bronze/raw_volume"
checkpoint_path =  "/Volumes/retail_catalog1/bronze/raw_volume/_checkpoint"

schema_hint = """
transaction_id STRING,
customer_id STRING,
amount DOUBLE,
store_id STRING,
timestamp STRING,
payment_type STRING
"""

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", True)
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.schemaHints", schema_hint)
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

# CLEAN IMMEDIATELY (IMPORTANT)
df_clean = df.select(
    col("transaction_id"),
    col("customer_id"),
    col("amount").cast("double"),
    col("store_id"),
    col("timestamp"),
    col("payment_type")
)

(
df_clean.writeStream
.format("delta")
.option("checkpointLocation", checkpoint_path)
.option("mergeSchema", "true")
.outputMode("append")
.trigger(availableNow=True) \
.table(f"{catalog}.{schema}.transactions")
)

In [0]:
%sql
SHOW VOLUMES IN retail_catalog1.bronze;

database,volume_name
bronze,raw_volume
